<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 05. Batch Normalization — Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift

- **원 논문:** [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://proceedings.mlr.press/v37/ioffe15.pdf)
- **저자 / 발표:** Sergey Ioffe and Christian Szegedy · ICML 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 90분
- **선수 지식:** 평균·분산, nn.Module parameters/buffers, train/eval mode

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

Algorithm 1의 mini-batch mean/variance, normalization, 학습 가능한 γ/β를 직접 구현하고 running statistics를 갖는 module로 확장한다. 표준화 수치와 train/eval 동작을 assert하며 서로 다른 feature scale의 합성 분류에서 loss curve를 비교한다.

**원 논문과 다른 것**

원 논문은 ImageNet의 Inception network와 MNIST에서 큰 batch training, population-statistic 추론, 여러 optimization schedule을 평가한다. 여기서는 96×4 tabular mini-batches와 16-unit MLP를 사용하며 논문의 claimed speedup 자체를 benchmark하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3, Algorithm 1 lines 1–2: μ_B와 σ_B² | `manual_batch_norm`의 mean과 biased batch variance | 정규화 결과의 feature-wise mean≈0, variance≈1 assert |
| §3, Eqs. (1)–(2)와 Algorithm 1 lines 3–4: x-hat, y=γx-hat+β | epsilon 정규화와 learnable `weight`/`bias` | 임의 γ/β를 넣은 출력 mean이 β인지 검증 |
| §3.1: inference에서 population statistics 사용 | `RunningBatchNorm1d.running_mean/running_var` buffers | eval 두 호출이 같고 running buffers가 변하지 않는지 assert |
| §3.2: BN transform 뒤 nonlinearity 적용 | `ScaledMLP`: Linear → BN → ReLU | 서로 다른 입력 scale에서 BN/no-BN loss와 정확도 plot |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\hat x_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}},
\qquad y_i=\gamma\hat x_i+\beta
$$

- $\mu_B,\sigma_B^2$는 mini-batch 통계, $\gamma,\beta$는 학습 가능한 affine parameter입니다.
- 학습 중에는 batch 통계, 평가 중에는 running 통계를 사용하는 차이가 핵심입니다.
- 직접 구한 normalization과 framework layer의 출력·gradient를 비교합니다.
- dense 입력은 feature 축, CNN 입력은 channel 축별로 통계를 계산합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3, Algorithm 1 lines 1–2: μ_B와 σ_B²
- **노트북 구현:** `manual_batch_norm`의 mean과 biased batch variance
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 정규화 결과의 feature-wise mean≈0, variance≈1 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 ImageNet의 Inception network와 MNIST에서 큰 batch training, population-statistic 추론, 여러 optimization schedule을 평가한다. 여기서는 96×4 tabular mini-batches와 16-unit MLP를 사용하며 논문의 claimed speedup 자체를 benchmark하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Algorithm 1을 네 줄로 분해하기

feature마다 mini-batch 평균 `μ_B`와 분산 `σ_B²`를 구하고, epsilon을 더해 정규화한
뒤 학습되는 `γ, β`로 표현력을 복원합니다. 이 실습의 batch variance는 Algorithm 1처럼
분모가 `m`인 biased variance입니다. 추론 때는 batch 자체가 아니라 학습 중 누적한
running statistics를 씁니다. 이는 단일 batch 함수와 stateful module을 따로 구현하는 이유입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,torch.randn,torch.tensor,ACCELERATOR.move,Tensor.mean,Tensor.std -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(5)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device

generator = torch.Generator().manual_seed(707)
latent_features = torch.randn(96, 4, generator=generator)
feature_scale = torch.tensor([1.0, 20.0, 0.05, 5.0])
feature_shift = torch.tensor([0.0, 3.0, -2.0, 10.0])
bn_x = latent_features * feature_scale + feature_shift
bn_y = (
    latent_features[:, 0]
    + latent_features[:, 1]
    - 0.8 * latent_features[:, 2]
    + 0.5 * latent_features[:, 3]
    > 0
).long()
bn_x, bn_y = ACCELERATOR.move(bn_x, bn_y)
assert bn_x.shape == (96, 4)
assert bn_y.unique().tolist() == [0, 1]
print(ACCELERATOR.summary())
print("raw means:", bn_x.mean(0).round(decimals=2).tolist())
print("raw stds: ", bn_x.std(0, unbiased=False).round(decimals=2).tolist())

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Algorithm 1 lines 1–2: μ_B와 σ_B² / §3, Eqs. (1)–(2)와 Algorithm 1 lines 3–4: x-hat, y=γx-hat+β
- **이 셀의 책임:** `manual_batch_norm`의 mean과 biased batch variance / epsilon 정규화와 learnable `weight`/`bias`
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 결과의 feature-wise mean≈0, variance≈1 assert / 임의 γ/β를 넣은 출력 mean이 β인지 검증. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.sqrt,torch.allclose,torch.zeros,torch.ones -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def manual_batch_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=0)
    variance = x.var(dim=0, unbiased=False)
    normalized = (x - mean) / torch.sqrt(variance + eps)
    return gamma * normalized + beta, mean, variance


gamma = torch.tensor([1.0, 0.5, 2.0, 1.5], device=DEVICE)
beta = torch.tensor([0.0, 1.0, -1.0, 0.25], device=DEVICE)
transformed, batch_mean, batch_variance = manual_batch_norm(bn_x, gamma, beta)
recovered_normalized = (transformed - beta) / gamma
assert torch.allclose(
    recovered_normalized.mean(0),
    torch.zeros(4, device=DEVICE),
    atol=1e-5,
)
assert torch.allclose(
    recovered_normalized.var(0, unbiased=False),
    torch.ones(4, device=DEVICE),
    atol=5e-3,
)
assert torch.allclose(transformed.mean(0), beta, atol=1e-5)
print("normalized mean:", recovered_normalized.mean(0).round(decimals=5).tolist())
print(
    "normalized var: ",
    recovered_normalized.var(0, unbiased=False).round(decimals=5).tolist(),
)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3, Algorithm 1 lines 1–2: μ_B와 σ_B² / §3, Eqs. (1)–(2)와 Algorithm 1 lines 3–4: x-hat, y=γx-hat+β / §3.1: inference에서 population statistics 사용
- **이 셀의 책임:** `manual_batch_norm`의 mean과 biased batch variance / epsilon 정규화와 learnable `weight`/`bias` / `RunningBatchNorm1d.running_mean/running_var` buffers
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 결과의 feature-wise mean≈0, variance≈1 assert / 임의 γ/β를 넣은 출력 mean이 β인지 검증 / eval 두 호출이 같고 running buffers가 변하지 않는지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Parameter,Module.register_buffer,torch.no_grad,Tensor.detach,Module.train,Module.eval,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Parameter(data, requires_grad=True)</code></summary>

#### `nn.Parameter(data, requires_grad=True)`

- **역할:** 텐서를 학습 파라미터로 등록합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.Parameter.html)

</details>

<details>
<summary><code>module.register_buffer(name, tensor, persistent=True)</code></summary>

#### `module.register_buffer(name, tensor, persistent=True)`

- **역할:** 학습하지 않지만 state와 장치 이동을 따르는 텐서를 등록합니다.
- **입력·반환:** 이름·텐서·저장 여부를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** position 상수나 running state를 파라미터와 구분해 보관합니다.
- **주의점:** 일반 attribute와 달리 state dict 포함 여부가 `persistent`에 좌우됩니다.
- **공식 문서:** [Module.register_buffer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class RunningBatchNorm1d(nn.Module):
    def __init__(self, features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(features))
        self.bias = nn.Parameter(torch.zeros(features))
        self.register_buffer("running_mean", torch.zeros(features))
        self.register_buffer("running_var", torch.ones(features))
        self.eps = eps
        self.momentum = momentum

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=0)
            variance = x.var(dim=0, unbiased=False)
            population_variance_estimate = x.var(dim=0, unbiased=True)
            with torch.no_grad():
                self.running_mean.lerp_(mean.detach(), self.momentum)
                self.running_var.lerp_(
                    population_variance_estimate.detach(),
                    self.momentum,
                )
        else:
            mean = self.running_mean
            variance = self.running_var
        normalized = (x - mean) / torch.sqrt(variance + self.eps)
        return self.weight * normalized + self.bias


running_bn = RunningBatchNorm1d(4).to(DEVICE)
running_bn.train()
train_output = running_bn(bn_x[:24])
assert torch.allclose(train_output.mean(0), torch.zeros(4, device=DEVICE), atol=1e-5)
saved_mean = running_bn.running_mean.clone()
saved_var = running_bn.running_var.clone()

running_bn.eval()
eval_one = running_bn(bn_x[:8])
eval_two = running_bn(bn_x[:8])
assert torch.equal(eval_one, eval_two)
assert torch.equal(saved_mean, running_bn.running_mean)
assert torch.equal(saved_var, running_bn.running_var)
print("running mean after one batch:", saved_mean.round(decimals=3).tolist())

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §3.2: BN transform 뒤 nonlinearity 적용
- **이 셀의 책임:** `ScaledMLP`: Linear → BN → ReLU
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 서로 다른 입력 scale에서 BN/no-BN loss와 정확도 plot. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,F.relu,torch.optim.SGD,Module.parameters,torch.arange,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,Optimizer.step,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 6개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>torch.optim.SGD(params, lr, momentum=0, weight_decay=0, ...)</code></summary>

#### `torch.optim.SGD(params, lr, momentum=0, weight_decay=0, ...)`

- **역할:** gradient 방향으로 파라미터를 이동하는 SGD optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률·momentum 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 고전 CNN과 최적화 기초 실험의 update 규칙을 명시합니다.
- **주의점:** momentum·weight decay·Nesterov 설정에 따라 실제 update 식이 달라집니다.

##### 관련 수식과 코드 연결

$$
g_t=\nabla_\theta L_t,\qquad v_t=\mu v_{t-1}+g_t,\qquad\theta_{t+1}=\theta_t-\eta v_t
$$

- **기호:** $\eta$는 학습률, $\mu$는 momentum 계수입니다. $\mu=0$이면 기본 SGD입니다.
- **수식 → 코드:** `momentum`이 0이면 현재 gradient만, 양수이면 이전 update 방향도 함께 사용합니다.
- **직관:** 손실이 가장 빠르게 증가하는 gradient의 반대 방향으로 parameter를 이동합니다.
- **shape 확인:** 각 momentum buffer는 대응 parameter와 같은 shape입니다.
- **공식 문서:** [torch.optim.SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class ScaledMLP(nn.Module):
    def __init__(self, use_batch_norm):
        super().__init__()
        self.hidden = nn.Linear(4, 16)
        self.normalizer = RunningBatchNorm1d(16) if use_batch_norm else nn.Identity()
        self.output = nn.Linear(16, 2)

    def forward(self, x):
        return self.output(F.relu(self.normalizer(self.hidden(x))))


def train_scaled_model(use_batch_norm):
    torch.manual_seed(808)
    model = ScaledMLP(use_batch_norm).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.03)
    history = []
    model.train()
    for step in range(48):
        start = (step * 24) % len(bn_x)
        batch_indices = torch.arange(start, start + 24, device=bn_x.device) % len(bn_x)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(bn_x[batch_indices]), bn_y[batch_indices])
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
    model.eval()
    with torch.no_grad():
        accuracy = (model(bn_x).argmax(1) == bn_y).float().mean().item()
    return model, history, accuracy


plain_model, plain_losses, plain_accuracy = train_scaled_model(False)
bn_model, bn_losses, bn_accuracy = train_scaled_model(True)
assert torch.isfinite(torch.tensor(plain_losses + bn_losses)).all()
assert min(bn_losses[1:]) < bn_losses[0]

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.2: BN transform 뒤 nonlinearity 적용
- **이 셀의 책임:** `ScaledMLP`: Linear → BN → ReLU
- **Tensor shape 계약:** batch [B,D] 또는 [B,C,H,W] → 같은 shape의 normalized activation
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 서로 다른 입력 scale에서 BN/no-BN loss와 정확도 plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
plt.figure(figsize=(6, 3.4))
plt.plot(plain_losses, label="without BN", alpha=0.85)
plt.plot(bn_losses, label="with Algorithm-1 BN", alpha=0.85)
plt.xlabel("mini-batch step")
plt.ylabel("cross-entropy")
plt.title("Optimization on differently scaled features")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"without BN: loss={plain_losses[-1]:.3f}, accuracy={plain_accuracy:.1%}")
print(f"with BN:    loss={bn_losses[-1]:.3f}, accuracy={bn_accuracy:.1%}")

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 ImageNet의 Inception network와 MNIST에서 큰 batch training, population-statistic 추론, 여러 optimization schedule을 평가한다. 여기서는 96×4 tabular mini-batches와 16-unit MLP를 사용하며 논문의 claimed speedup 자체를 benchmark하지 않는다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.